# 📈 Kredi Temerrüt Olasılıklarının Makine Öğrenmesi ile Tahmin Edilmesi

Bu notebook'ta, **Alman Kredi Veri Setini (German Credit Dataset)** analiz edeceğiz ve her bir müşterinin kredi temerrüt (default) olasılığını ($p_i$) hesaplamak için klasik bir makine öğrenmesi sınıflandırma modeli (XGBoost veya Random Forest) eğiteceğiz.

Bu olasılıklar, bir sonraki aşamada **Kuantum Genlik Tahmini (QAE)** algoritmasında kuantum belirsizlik modeline girdi olarak beslenecektir.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, roc_curve

# Veriyi yükleyelim
data_path = os.path.join("..", "data", "raw", "german_credit_data.csv")
df = pd.read_csv(data_path)
print(f"Veri Boyutu: {df.shape}")
df.head()

## 🧹 Veri Ön İşleme (Preprocessing)

Alman kredi veri setinde hem nümerik hem de kategorik (text) sütunlar bulunmaktadır. Kategorik sütunları `LabelEncoder` ile sayısal değerlere dönüştüreceğiz ve nümerik değerleri ölçeklendireceğiz.

Hedef değişkenimiz `credit_risk` sütunudur:
*   `1`: İyi kredi (Temerrüt etmeyen - Non-default)
*   `0`: Kötü kredi (Temerrüt eden - Default)

Bizim amacımız **temerrüt olasılığını** tahmin etmek olduğundan, hedef etiketimizi tersine çevireceğiz (1: Default, 0: Non-default). Bu notebook'ta hedef değişkeni doğrudan `default` olarak tanımlayacağız (`default = 1 - credit_risk`).

In [ ]:
# Kategorik sütunları bulalım
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
num_cols = df.select_dtypes(exclude=['object']).columns.tolist()
num_cols.remove('credit_risk')

print("Kategorik Sütunlar:", cat_cols)
print("Sayısal Sütunlar:", num_cols)

# Kategorik değişkenleri kodlayalım
df_processed = df.copy()
label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Hedef değişkeni 'default' olarak tanımlayalım (1: Temerrüt, 0: Normal)
df_processed['default'] = 1 - df_processed['credit_risk']
df_processed.drop(columns=['credit_risk'], inplace=True)

# Özellikler ve Hedef Değişkeni ayıralım
X = df_processed.drop(columns=['default'])
y = df_processed['default']

# Veriyi Eğitim ve Test olarak bölelim
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Eğitim Kümesi: {X_train.shape}, Test Kümesi: {X_test.shape}")

## 🤖 XGBoost Model Eğitimi ve Performans Analizi

Kredi temerrüt riskini tahmin etmek için yaygın olarak tercih edilen güçlü bir gradyan artırma algoritması olan **XGBoost Classifier** modelini eğitiyoruz. Modeli eğittikten sonra test kümesindeki performansını inceleyeceğiz ve her müşteri için tahmin edilen olasılıkları (`predict_proba`) elde edeceğiz.

In [ ]:
# XGBoost modelini tanımlayalım ve eğitelim
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.08,
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)
model.fit(X_train, y_train)

# Test kümesinde tahmin yapalım
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1] # Temerrüt (default=1) olasılığı

# Raporlayalım
print("Sınıflandırma Raporu:")
print(classification_report(y_test, y_pred))
print(f"ROC AUC Skoru: {roc_auc_score(y_test, y_prob):.4f}")

# Özellik önem düzeyleri
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)
print("\nEn Önemli 5 Özellik:")
print(importance.head(5))

In [ ]:
# Kredi portföyümüz için rastgele 3 veya 4 kredi seçelim ve bu kredilerin
# temerrüt olasılıkları ile kayıp miktarlarını belirleyelim (Kuantum QAE için)
# (QAE donanım limitleri nedeniyle küçük kübit sayıları için küçük portföyler seçilir)
np.random.seed(42)
portfolio_indices = np.random.choice(X_test.index, size=3, replace=False)

portfolio_df = pd.DataFrame({
    'Client_ID': portfolio_indices,
    'Default_Probability': model.predict_proba(X.loc[portfolio_indices])[:, 1],
    'Loss_Given_Default': [100000, 250000, 150000] # Her bir kredinin tutarı
})

# Kaydedelim
processed_dir = os.path.join("..", "data", "processed")
os.makedirs(processed_dir, exist_ok=True)
portfolio_df.to_csv(os.path.join(processed_dir, "predicted_portfolio.csv"), index=False)

print("Kuantum Portföy Verileri Başarıyla Kaydedildi:")
portfolio_df